   # Structured-Answer LoRA
   Fine-tuning Qwen2.5-0.5B-Instruct with LoRA on a free T4.
   Result: format compliance 39/40 vs 0/40 for the base model.
   Model: https://huggingface.co/Meow07/qwen2.5-0.5b-format-lora
   Demo: https://huggingface.co/spaces/Meow07/format-lora-demo

In [ ]:
!pip install -q -U transformers peft trl datasets accelerate bitsandbytes

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [ ]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
from datasets import load_dataset
raw = load_dataset("HuggingFaceH4/no_robots", split="train")
qs = [p for p in raw["prompt"] if 20 < len(p) < 300][:450]
print(len(qs)); print(qs[0])

450
Help write a letter of 100 -200 words to my future self for Kyra, reflecting on her goals and aspirations.


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tid = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(tid)
tok.padding_side = "left"
teacher = AutoModelForCausalLM.from_pretrained(tid, dtype=torch.float16, device_map="auto")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
SYSTEM = ("You are a helpful assistant. ALWAYS reply in exactly this format and nothing else:\n"
          "Summary: <one sentence>\nKey points:\n- <point>\n- <point>\n- <point>\n"
          "Next step: <one sentence>")

answers = []
for i in range(0, len(qs), 16):
    batch = qs[i:i+16]
    texts = [tok.apply_chat_template(
                [{"role":"system","content":SYSTEM},{"role":"user","content":q}],
                add_generation_prompt=True, tokenize=False) for q in batch]
    inp = tok(texts, return_tensors="pt", padding=True).to(teacher.device)
    with torch.no_grad():
        out = teacher.generate(**inp, max_new_tokens=220, do_sample=True,
                               temperature=0.3, pad_token_id=tok.eos_token_id)
    for o in out:
        answers.append(tok.decode(o[inp["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    print(i, end=" ")

0 16 32 48 64 80 96 112 128 144 160 176 192 208 224 240 256 272 288 304 320 336 352 368 384 400 416 432 448 

In [ ]:
import re
def norm(a):
    a = a.replace("**", "").strip()
    a = re.sub(r"(?i)key points\s*:", "Key points:", a)
    a = re.sub(r"(?i)next step\s*:", "Next step:", a)
    a = re.sub(r"(?i)^summary\s*:", "Summary:", a)
    return a

reasons = {"no_summary":0, "no_keypoints":0, "no_nextstep":0, "ok":0}
good = []
for q, a in zip(qs, answers):
    n = norm(a)
    if not n.startswith("Summary:"): reasons["no_summary"] += 1
    elif "Key points:" not in n:     reasons["no_keypoints"] += 1
    elif "Next step:" not in n:      reasons["no_nextstep"] += 1
    else:
        reasons["ok"] += 1
        good.append((q, n))
print(reasons)

bad = [a for a in answers if "Next step:" not in norm(a)][:2]
for b in bad: print("-----\n", b[-300:])

{'no_summary': 289, 'no_keypoints': 4, 'no_nextstep': 12, 'ok': 145}
-----
 short not to live it to the fullest, so make each day count by being present, kind, and compassionate.

May your future be filled with joy, fulfillment, and all the blessings you deserve. Stay curious, keep dreaming big, and never stop believing in yourself.

With love and admiration,
Your Past Self
-----
 "Family unity on the biggest day of the year! 🌟 #Easter2023 #FamilyFirst #PhotoMoments"


In [ ]:
raw = load_dataset("HuggingFaceH4/no_robots", split="train")
used = set(qs)
pool = [r["prompt"] for r in raw
        if r["category"] in ("Open QA", "Brainstorm", "Chat", "Closed QA")
        and 20 < len(r["prompt"]) < 300 and r["prompt"] not in used][:450]
print("new questions:", len(pool))

REMINDER = "\n\n(Reply ONLY in this format: Summary / Key points (3 bullets) / Next step.)"
new_answers = []
for i in range(0, len(pool), 16):
    batch = pool[i:i+16]
    texts = [tok.apply_chat_template(
                [{"role":"system","content":SYSTEM},{"role":"user","content":q + REMINDER}],
                add_generation_prompt=True, tokenize=False) for q in batch]
    inp = tok(texts, return_tensors="pt", padding=True).to(teacher.device)
    with torch.no_grad():
        out = teacher.generate(**inp, max_new_tokens=260, do_sample=True,
                               temperature=0.3, pad_token_id=tok.eos_token_id)
    for o in out:
        new_answers.append(tok.decode(o[inp["input_ids"].shape[1]:], skip_special_tokens=True).strip())
    print(i, end=" ")

for q, a in zip(pool, new_answers):
    n = norm(a)
    if n.startswith("Summary:") and "Key points:" in n and "Next step:" in n:
        good.append((q, n))
print("\ntotal good:", len(good))

new questions: 450
0 16 32 48 64 80 96 112 128 144 160 176 192 208 224 240 256 272 288 304 320 336 352 368 384 400 416 432 448 
total good: 563
Steve is a chatbot that is nihilistic and doesn't see a point in answering questions but still answers.
Summary: Steve, the chatbot, provides responses without considering their value or purpose.
  
Key points:
1. Steve's behavior disregards the context of user inquiries.
2. Despite its lack of engagement with questions, it continues to respond.
3. The bot operates purely on providing information without moral judgment.

Next step: Implement mechanisms to enhance Steve's understanding of user intent and provide more meaningful responses.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import re, json, random

def fix(a):
    lines = [l.rstrip() for l in a.splitlines() if l.strip()]
    if len(lines) != 6: return None
    if not lines[0].startswith("Summary:") or lines[1].strip() != "Key points:": return None
    if not lines[5].startswith("Next step:"): return None
    bullets = []
    for l in lines[2:5]:
        if not re.match(r"^\s*(?:[-*•]|\d+[.)])\s+\S", l): return None
        bullets.append("- " + re.sub(r"^\s*(?:[-*•]|\d+[.)])\s*", "", l))
    return "\n".join([lines[0], "Key points:", *bullets, lines[5]])

seen, clean = set(), []
for q, a in good:
    f = fix(a)
    if f and q not in seen:
        seen.add(q); clean.append({"messages":[{"role":"user","content":q},
                                               {"role":"assistant","content":f}]})
random.seed(42); random.shuffle(clean)
n_val = max(30, len(clean)//10)
val, train = clean[:n_val], clean[n_val:]
print(len(clean), "clean |", len(train), "train |", len(val), "val")
json.dump(train, open("train.json","w")); json.dump(val, open("val.json","w"))
print(train[0]["messages"][1]["content"])

488 clean | 440 train | 48 val
Summary: To bond with your inactive cat, consider interactive toys like puzzle feeders or laser pointers.
Key points:
- Interactive toys keep cats mentally stimulated.
- Laser pointers provide entertainment without requiring physical activity.
- Puzzle feeders offer challenges that engage the cat's brain.
Next step: Purchase or create an interactive toy for your cat to play with regularly.


In [ ]:
from google.colab import files
files.download("train.json"); files.download("val.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import gc, torch
del teacher
gc.collect(); torch.cuda.empty_cache()

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

ds = load_dataset("json", data_files={"train":"train.json","validation":"val.json"})
mid = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(mid)
model = AutoModelForCausalLM.from_pretrained(mid, dtype=torch.float32, device_map="auto")

TESTS = ["How do vaccines work?",
         "What are some good ways to prepare for a job interview?",
         "Why is the sky blue?"]

def ask(m, q):
    text = tok.apply_chat_template([{"role":"user","content":q}], add_generation_prompt=True, tokenize=False)
    inp = tok(text, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inp, max_new_tokens=250, do_sample=False,
                         repetition_penalty=1.05, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

print("=== BEFORE ===")
for q in TESTS: print("Q:", q, "\n", ask(model, q), "\n")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

=== BEFORE ===
Q: How do vaccines work? 
 Vaccines work by introducing a harmless piece of the pathogen (the germ that causes an illness) into the body through a vaccine. This harmless piece is designed to stimulate the immune system to recognize and fight off the actual pathogen. When the immune system recognizes the pathogen, it produces antibodies that can neutralize or destroy the pathogen. This process is called immunization.

The immune system then mounts an immune response against the actual pathogen, which helps protect the individual from contracting the illness. The immune response is usually temporary, as the body will eventually eliminate the pathogen through natural processes such as phagocytosis (the engulfing of the pathogen by white blood cells), cell-mediated immunity, and antibody production.

Vaccines are typically made in a laboratory and are administered to individuals through various routes, including injection, oral ingestion, or inhalation. They are designed to 

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
args = SFTConfig(output_dir="out", per_device_train_batch_size=4, gradient_accumulation_steps=4,
                 learning_rate=2e-4, num_train_epochs=2, logging_steps=5,
                 eval_strategy="steps", eval_steps=10, fp16=True, max_length=512, report_to="none")
trainer = SFTTrainer(model=model, args=args, train_dataset=ds["train"],
                     eval_dataset=ds["validation"], peft_config=lora, processing_class=tok)
trainer.train()

Tokenizing train dataset:   0%|          | 0/440 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/440 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/440 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/440 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/48 [00:00<?, ? examples/s]

Step,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
10,1.373871,1.266936,1.356438,0.688070,23110.000000
20,1.246028,1.205095,1.310451,0.700223,46103.000000
30,1.201245,1.187938,1.255621,0.701764,68276.000000
40,1.106053,1.182186,1.158105,0.703471,91665.000000
50,1.041398,1.174560,1.197308,0.706822,114740.000000
56,1.039588,1.172804,1.193201,0.706816,127070.000000


TrainOutput(global_step=56, training_loss=1.238999037870339, metrics={'train_runtime': 108.0545, 'train_samples_per_second': 8.144, 'train_steps_per_second': 0.518, 'total_flos': 336556816066560.0, 'train_loss': 1.238999037870339, 'epoch': 2.0})

In [ ]:
trainer.model.save_pretrained("format-lora"); tok.save_pretrained("format-lora")

m = trainer.model
m.eval(); m.config.use_cache = True
if hasattr(m, "gradient_checkpointing_disable"): m.gradient_checkpointing_disable()

print("=== AFTER ===")
for q in TESTS: print("Q:", q, "\n", ask(m, q), "\n")

=== AFTER ===
Q: How do vaccines work? 
 Summary: Vaccines stimulate the immune system to recognize and fight off specific pathogens.
Key points:
- They contain weakened or dead versions of pathogens.
- The body's immune cells recognize these as foreign invaders.
- When injected into the body, they trigger an immune response.
Next step: Learn more about how vaccines work in detail. 

Q: What are some good ways to prepare for a job interview? 
 Summary: Prepare thoroughly by researching the company, practicing common interview questions, and being well-prepared for any potential questions.
Key points:
- Research the company thoroughly.
- Practice common interview questions.
- Be well-prepared for any potential questions.
Next step: Start preparing now by gathering all necessary materials such as简历, references, and any relevant documents. 

Q: Why is the sky blue? 
 Summary: The sky appears blue because it reflects sunlight through Earth's atmosphere.
Key points:
- The Earth's atmosphere

In [ ]:
import re
from datasets import load_dataset
raw = load_dataset("HuggingFaceH4/no_robots", split="test")
trained = set(ds["train"]["messages"][i][0]["content"] for i in range(len(ds["train"])))
evalq = [r["prompt"] for r in raw
         if r["category"] in ("Open QA","Brainstorm","Chat","Closed QA")
         and 20 < len(r["prompt"]) < 300 and r["prompt"] not in trained][:40]

def ok(a):
    return bool(re.match(r"^Summary:.*\nKey points:\n- .*\n- .*\n- .*\nNext step:", a.strip(), re.S))

def rate(model_):
    hits = sum(ok(ask(model_, q)) for q in evalq)
    return hits, len(evalq)

print("fine-tuned:", rate(m))
with m.disable_adapter():
    print("base model:", rate(m))

fine-tuned: (39, 40)
base model: (0, 40)


In [ ]:
from huggingface_hub import login, HfApi
login()   # paste your Write token when asked

user = HfApi().whoami()["name"]
repo = f"{user}/qwen2.5-0.5b-format-lora"
m.push_to_hub(repo)
tok.push_to_hub(repo)
print("https://huggingface.co/" + repo)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 38.0kB / 35.2MB            

Validating                    :           |   0%            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpt7u6a9kp/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Validating                    :           |   0%            

https://huggingface.co/Meow07/qwen2.5-0.5b-format-lora
